# DCE-MRI — primary model training

Resumable training of one architecture and one random seed per run, with validation-based checkpoint and threshold selection.


In [ ]:

from pathlib import Path
import os, sys, json, time, math, random, hashlib, zipfile, shutil, warnings
warnings.filterwarnings("ignore")

RUN_PRECHECK = True
RUN_VISUALIZATION = True
RUN_TRAINING = True  # Set False for pre-check only.
RUN_EVALUATION_ONLY = False  # True only if a best checkpoint already exists.
CREATE_LIGHT_ZIP = True
CREATE_RESUME_ZIP = True

# Run one model/seed configuration per long training session.
EXPERIMENT_MODEL = "swin_tiny_unet"
EXPERIMENT_SEED = 42

MODELS_TO_RUN = [EXPERIMENT_MODEL]
SEEDS = [EXPERIMENT_SEED]

# Optional resume input path from a prior Kaggle run.
# Example: RESUME_INPUT_DIRS = ["/kaggle/input/trackb-swin-seed42-resume/ROI_MRI_TrackB_Phase2_RESULTS"]
RESUME_INPUT_DIRS = []

# Keep the runtime guard below the Kaggle session limit.
AUTO_STOP_BEFORE_TIMEOUT = True
STOP_AFTER_MINUTES = 650  # about 10 h 50 min; leave margin for final writing/zipping

IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 0  # Keep the conservative worker setting for large NPZ collections; increase only after a successful run.
MAX_EPOCHS = 80
PATIENCE = 15
WARMUP_EPOCHS = 5
GRAD_CLIP = 1.0
WEIGHT_DECAY = 1e-4
LR_DECODER = 2e-4
LR_ENCODER = 2e-5
THRESHOLDS = [round(x, 2) for x in [0.05 + 0.05*i for i in range(19)]]

# Keep debug limits as None for full runs.
DEBUG_MAX_TRAIN = None
DEBUG_MAX_VAL = None
DEBUG_MAX_TEST = None
DEBUG_MAX_EXTERNAL = None

USE_GOOGLE_DRIVE = False  # Use /kaggle/input on Kaggle; enable the Colab path mode only for Colab.
MOUNT_DRIVE = False

COLAB_DATA_ROOT = Path("/content/drive/MyDrive/ROI_MRI_Crops_256_v1")
KAGGLE_DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/roi-mri-crops-256-v1/ROI_MRI_Crops_256_v1"),
    Path("/kaggle/input/roi-mri-crops-256-v1"),
    Path("/kaggle/input/ROI_MRI_Crops_256_v1"),
    Path("/kaggle/input"),
]

OUT_DIR = Path("/kaggle/working/ROI_MRI_TrackB_Phase2_RESULTS") if Path("/kaggle/working").exists() else Path("/content/ROI_MRI_TrackB_Phase2_RESULTS")
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
METRIC_DIR = OUT_DIR / "metrics"
LOG_DIR = OUT_DIR / "logs"
FIG_DIR = OUT_DIR / "figures"
PROB_DIR = OUT_DIR / "probabilities"

for p in [OUT_DIR, CHECKPOINT_DIR, METRIC_DIR, LOG_DIR, FIG_DIR, PROB_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RUN_STARTED_AT = time.time()

print("Configuration")
print("EXPERIMENT_MODEL:", EXPERIMENT_MODEL)
print("EXPERIMENT_SEED:", EXPERIMENT_SEED)
print("OUT_DIR:", OUT_DIR)
print("AUTO_STOP_BEFORE_TIMEOUT:", AUTO_STOP_BEFORE_TIMEOUT, "STOP_AFTER_MINUTES:", STOP_AFTER_MINUTES)


## Step 1 — Drive mounting / path detection


In [ ]:

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and USE_GOOGLE_DRIVE and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

from pathlib import Path
import pandas as pd

def find_data_root():
    candidates = []
    if IN_COLAB and USE_GOOGLE_DRIVE:
        candidates.append(COLAB_DATA_ROOT)
    candidates.extend(KAGGLE_DATA_ROOT_CANDIDATES)
    candidates.append(Path("/content/ROI_MRI_Crops_256_v1"))
    candidates.append(Path("/content/drive/MyDrive/ROI_MRI_Crops_256_v1"))

    for root in candidates:
        root = Path(root)
        if (root / "roi_mri_manifest.csv").exists() and (root / "npz").exists():
            return root

    search_roots = [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]
    for sr in search_roots:
        if sr.exists():
            hits = list(sr.rglob("roi_mri_manifest.csv"))
            for h in hits:
                root = h.parent
                if (root / "npz").exists():
                    return root

    raise FileNotFoundError(
        "Could not find ROI_MRI_Crops_256_v1 with roi_mri_manifest.csv and npz/ folder. "
        "Add the dataset as a Kaggle input or mount it in Colab."
    )

DATA_ROOT = find_data_root()
MANIFEST_PATH = DATA_ROOT / "roi_mri_manifest.csv"
NPZ_ROOT = DATA_ROOT / "npz"

print("DATA_ROOT:", DATA_ROOT)
print("MANIFEST_PATH:", MANIFEST_PATH)
print("NPZ_ROOT:", NPZ_ROOT)

df = pd.read_csv(MANIFEST_PATH)
print("Manifest rows:", len(df))
display(df.head())


## Step 2 — Imports and determinism


In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

try:
    import cv2
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"])
    import cv2

try:
    import albumentations as A
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "albumentations"])
    import albumentations as A

from scipy.ndimage import binary_erosion, distance_transform_edt
from tqdm.auto import tqdm

try:
    from torchvision.models import swin_t
    TORCHVISION_OK = True
except Exception as e:
    TORCHVISION_OK = False
    print("Torchvision Swin unavailable:", e)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def seed_everything(seed:int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


## Step 3 — Manifest loading and split audit


In [ ]:

import os, json, re
from pathlib import Path
import pandas as pd
import numpy as np

df = pd.read_csv(MANIFEST_PATH)
print('Manifest rows:', len(df))
print('Columns:', list(df.columns))
display(df.head())

required_cols = ['sample_id', 'dataset', 'source', 'split', 'patient_id', 'npz_path', 'spacing_mm', 'crop_size_native']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns in the manifest: {missing}')

counts = df.groupby(['dataset', 'split']).size().reset_index(name='n_crops')
patients = df.groupby(['dataset', 'split'])['patient_id'].nunique().reset_index(name='n_patients')
display(counts)
display(patients)

splits = {s: set(df.loc[df['split'] == s, 'patient_id'].astype(str)) for s in ['train', 'validation', 'test', 'external']}
leaks = {
    'train_vs_validation': len(splits['train'] & splits['validation']),
    'train_vs_test': len(splits['train'] & splits['test']),
    'validation_vs_test': len(splits['validation'] & splits['test']),
    'dev_vs_external': len((splits['train'] | splits['validation'] | splits['test']) & splits['external']),
}
print('Patient leakage:', leaks)
if any(v > 0 for v in leaks.values()):
    raise RuntimeError(f'Patient leakage detected: {leaks}')

print('Building ultra-robust recursive NPZ index...')

search_roots = []
if NPZ_ROOT.exists():
    search_roots.append(NPZ_ROOT)
if DATA_ROOT.exists():
    search_roots.append(DATA_ROOT)

for candidate in [
    Path('/content/drive/MyDrive/ROI_MRI_Crops_256_v1'),
    Path('/content/drive/MyDrive'),
    Path('/content'),
    Path('/kaggle/input'),
]:
    if candidate.exists() and candidate not in search_roots:
        search_roots.append(candidate)

all_npz_files = []
seen = set()
for root in search_roots:
    try:
        files = list(root.rglob('*.npz'))
    except Exception as e:
        print('Could not scan', root, ':', e)
        files = []
    for p in files:
        sp = str(p.resolve()) if p.exists() else p.as_posix()
        if sp not in seen:
            seen.add(sp)
            all_npz_files.append(Path(p))

print('Search roots:')
for r in search_roots:
    print(' -', r)
print('Total recursive NPZ files found across roots:', len(all_npz_files))

if len(all_npz_files) == 0:
    raise FileNotFoundError(
        'No NPZ files found. Check that Google Drive is mounted and ROI_MRI_Crops_256_v1/npz exists.'
    )

NPZ_INDEX = {}
NPZ_INDEX_NAME = {}

def norm_key(x):
    return str(x).replace('\\', '/').strip().lower()

for p in all_npz_files:
    p = Path(p)
    keys = set()

    keys.add(p.as_posix())

    for root in [DATA_ROOT, NPZ_ROOT, Path('/content/drive/MyDrive/ROI_MRI_Crops_256_v1'), Path('/content/drive/MyDrive')]:
        try:
            if root.exists():
                keys.add(p.relative_to(root).as_posix())
        except Exception:
            pass

    sp = p.as_posix()
    sp_low = sp.lower()
    idx = sp_low.find('/npz/')
    if idx >= 0:
        keys.add(sp[idx+1:])
        keys.add(sp[idx+5:])

    if len(p.parts) >= 3:
        keys.add('/'.join(p.parts[-3:]))
    if len(p.parts) >= 4:
        keys.add('/'.join(p.parts[-4:]))

    for k in keys:
        NPZ_INDEX[norm_key(k)] = p

    NPZ_INDEX_NAME.setdefault(p.name.lower(), []).append(p)

print('NPZ index path keys:', len(NPZ_INDEX))
print('NPZ index filenames:', len(NPZ_INDEX_NAME))

manifest_rel = df['npz_path'].astype(str).str.replace('\\', '/', regex=False).str.strip()
manifest_names = set(Path(x).name.lower() for x in manifest_rel.dropna())
actual_names = set(NPZ_INDEX_NAME.keys())
name_overlap = len(manifest_names & actual_names)
print(f'Manifest/NPZ filename overlap: {name_overlap}/{len(manifest_names)} ({name_overlap/max(1,len(manifest_names)):.3f})')

def resolve_npz_path(rel):
    rel = str(rel).replace('\\', '/').strip()
    rel_lower = rel.lower()
    name_lower = Path(rel).name.lower()

    candidates = [
        Path(rel),
        DATA_ROOT / rel,
        NPZ_ROOT / rel,
        DATA_ROOT / 'npz' / rel,
        NPZ_ROOT / Path(rel).name,
    ]
    for p in candidates:
        try:
            if p.exists():
                return Path(p)
        except Exception:
            pass

    variants = {
        rel_lower,
        rel_lower.lstrip('/'),
        'npz/' + rel_lower if not rel_lower.startswith('npz/') else rel_lower,
        rel_lower[4:] if rel_lower.startswith('npz/') else rel_lower,
    }

    for key in variants:
        if key in NPZ_INDEX:
            return NPZ_INDEX[key]

    idx = rel_lower.find('npz/')
    if idx >= 0:
        suffix = rel_lower[idx:]
        if suffix in NPZ_INDEX:
            return NPZ_INDEX[suffix]
        suffix2 = rel_lower[idx+4:]
        if suffix2 in NPZ_INDEX:
            return NPZ_INDEX[suffix2]

    if name_lower in NPZ_INDEX_NAME:
        candidates_by_name = NPZ_INDEX_NAME[name_lower]
        if len(candidates_by_name) == 1:
            return candidates_by_name[0]
        # Try to disambiguate by split/dataset tokens from rel
        rel_tokens = set(Path(rel_lower).parts)
        scored = []
        for cand in candidates_by_name:
            cand_tokens = set(Path(cand.as_posix().lower()).parts)
            scored.append((len(rel_tokens & cand_tokens), cand))
        scored.sort(reverse=True, key=lambda x: x[0])
        if scored and scored[0][0] > 0:
            return scored[0][1]

    return None

sample_paths = df['npz_path'].sample(n=min(50, len(df)), random_state=42).tolist()
missing_sample = [p for p in sample_paths if resolve_npz_path(p) is None]

if missing_sample:
    print('\nERROR: Some sampled manifest paths cannot be resolved.')
    print('DATA_ROOT:', DATA_ROOT)
    print('NPZ_ROOT:', NPZ_ROOT)
    print('First manifest npz_path examples:')
    print(df['npz_path'].head(10).to_string(index=False))
    print('\nFirst actual NPZ examples:')
    for p in all_npz_files[:10]:
        print(' -', p)
    print('\nMissing sample examples:')
    for m in missing_sample[:10]:
        print(' -', m, '| filename in index:', Path(str(m)).name.lower() in actual_names)

    diag = pd.DataFrame({
        'npz_path': sample_paths,
        'resolved': [resolve_npz_path(p) is not None for p in sample_paths],
        'filename': [Path(str(p)).name for p in sample_paths],
        'filename_in_npz_index': [Path(str(p)).name.lower() in actual_names for p in sample_paths],
    })
    diag_path = LOG_DIR / 'npz_path_resolution_diagnostic.csv'
    diag.to_csv(diag_path, index=False)
    print('\nSaved diagnostic:', diag_path)
    raise FileNotFoundError('Sample NPZ files not found. See logs/npz_path_resolution_diagnostic.csv')

print('OK: sampled NPZ paths found.')

if RUN_PRECHECK:
    print('Checking all manifest NPZ paths for existence...')
    resolved_paths = df['npz_path'].apply(resolve_npz_path)
    exist_flags = resolved_paths.notna()
    n_missing = int((~exist_flags).sum())
    print(f'Manifest NPZ path existence: {int(exist_flags.sum())}/{len(df)} found; missing={n_missing}')
    if n_missing > 0:
        missing_df = df.loc[~exist_flags, ['sample_id','dataset','split','npz_path']].head(200)
        missing_df.to_csv(LOG_DIR / 'missing_npz_paths_head200.csv', index=False)
        raise FileNotFoundError(f'{n_missing} manifest NPZ paths cannot be resolved. See logs/missing_npz_paths_head200.csv')
else:
    resolved_paths = df['npz_path'].apply(resolve_npz_path)

df_abs = df.copy()
df_abs['npz_path_original'] = df_abs['npz_path']
df_abs['npz_path'] = resolved_paths.apply(lambda p: str(p))
abs_manifest_path = LOG_DIR / 'roi_mri_manifest_absolute_paths.csv'
df_abs.to_csv(abs_manifest_path, index=False)
print('Saved absolute-path manifest:', abs_manifest_path)

# Use absolute paths after dataset resolution.
df = df_abs.copy()

counts.to_csv(LOG_DIR / 'counts_by_dataset_split.csv', index=False)
patients.to_csv(LOG_DIR / 'patients_by_dataset_split.csv', index=False)
with open(LOG_DIR / 'split_leakage_audit.json', 'w', encoding='utf-8') as f:
    json.dump(leaks, f, indent=2, ensure_ascii=False)

## Step 4 — PyTorch dataset + light augmentation


In [ ]:


train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.ShiftScaleRotate(
        shift_limit=0.04,
        scale_limit=0.06,
        rotate_limit=10,
        border_mode=cv2.BORDER_CONSTANT,
        value=0,
        mask_value=0,
        interpolation=cv2.INTER_LINEAR,
        p=0.6,
    ),
])

class ROIMRIDataset(Dataset):
    def __init__(self, frame, data_root, augment=None, limit=None):
        self.df = frame.reset_index(drop=True).copy()
        if limit is not None:
            self.df = self.df.iloc[:limit].reset_index(drop=True)
        self.data_root = Path(data_root)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def resolve_path(self, rel):
        p = resolve_npz_path(rel)
        if p is not None:
            return p
        raise FileNotFoundError(f"NPZ not found: {rel}")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = self.resolve_path(row["npz_path"])
        z = np.load(path)
        image = z["image"].astype(np.float32)
        mask = z["mask"].astype(np.uint8)

        if image.ndim != 3 or image.shape[-1] != 3:
            raise ValueError(f"image must be H,W,3, received {image.shape} for {path}")
        if mask.ndim != 2:
            mask = np.squeeze(mask)
        image = np.clip(image, 0, 1)
        mask = (mask > 0).astype(np.uint8)

        if self.augment is not None:
            aug = self.augment(image=(image*255).astype(np.uint8), mask=mask)
            image = aug["image"].astype(np.float32) / 255.0
            mask = (aug["mask"] > 0).astype(np.uint8)

        x = torch.from_numpy(image).permute(2,0,1).float()
        y = torch.from_numpy(mask[None, ...]).float()

        meta = {
            "sample_id": str(row["sample_id"]),
            "split": str(row["split"]),
            "dataset": str(row["dataset"]),
            "patient_id": str(row["patient_id"]),
            "npz_path": str(row["npz_path"]),
            "spacing_mm": str(row.get("spacing_mm", "1;1;1")),
            "crop_size_native": float(row.get("crop_size_native", 256)),
        }
        return {"image": x, "mask": y, "meta": meta}


def make_split_df(split, limit=None):
    sub = df[df["split"] == split].copy().reset_index(drop=True)
    if limit is not None:
        sub = sub.iloc[:limit].reset_index(drop=True)
    return sub

train_df = make_split_df("train", DEBUG_MAX_TRAIN)
val_df = make_split_df("validation", DEBUG_MAX_VAL)
test_df = make_split_df("test", DEBUG_MAX_TEST)
ext_df = make_split_df("external", DEBUG_MAX_EXTERNAL)

print(len(train_df), len(val_df), len(test_df), len(ext_df))

train_ds = ROIMRIDataset(train_df, DATA_ROOT, augment=train_aug)
val_ds = ROIMRIDataset(val_df, DATA_ROOT, augment=None)
test_ds = ROIMRIDataset(test_df, DATA_ROOT, augment=None)
ext_ds = ROIMRIDataset(ext_df, DATA_ROOT, augment=None)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
ext_loader = DataLoader(ext_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

batch = next(iter(val_loader))
print("Batch image:", batch["image"].shape, batch["image"].min().item(), batch["image"].max().item())
print("Batch mask:", batch["mask"].shape, batch["mask"].sum().item())


## Step 5 — DCE quality-control visualization


In [ ]:


def overlay_rgb_on_gray(gray, mask, alpha=0.45):
    gray_u8 = (np.clip(gray,0,1)*255).astype(np.uint8)
    rgb = cv2.cvtColor(gray_u8, cv2.COLOR_GRAY2RGB)
    over = rgb.copy()
    over[mask > 0] = [255, 0, 0]
    return cv2.addWeighted(over, alpha, rgb, 1-alpha, 0)

if RUN_VISUALIZATION:
    rng = np.random.default_rng(42)
    examples = []
    for split_name, frame in [("train", train_df), ("validation", val_df), ("test", test_df), ("external", ext_df)]:
        if len(frame) == 0: continue
        row = frame.sample(1, random_state=42).iloc[0]
        path = resolve_npz_path(row["npz_path"])
        z = np.load(path)
        img = z["image"].astype(np.float32)
        msk = (z["mask"] > 0).astype(np.uint8)
        examples.append((split_name, row["sample_id"], img, msk))

    n = len(examples)
    fig, axes = plt.subplots(n, 6, figsize=(18, 3.2*n))
    if n == 1: axes = np.expand_dims(axes, 0)
    for r, (split_name, sid, img, msk) in enumerate(examples):
        titles = ["pre", "early", "late"]
        for c in range(3):
            axes[r,c].imshow(img[...,c], cmap="gray")
            axes[r,c].set_title(f"{split_name} {titles[c]}")
            axes[r,c].axis("off")
        axes[r,3].imshow(msk, cmap="gray")
        axes[r,3].set_title("mask")
        axes[r,3].axis("off")
        axes[r,4].imshow(overlay_rgb_on_gray(img[...,1], msk))
        axes[r,4].set_title("early overlay")
        axes[r,4].axis("off")
        lesion_means = [float(img[...,c][msk>0].mean()) for c in range(3)]
        axes[r,5].plot([0,1,2], lesion_means, marker="o")
        axes[r,5].set_xticks([0,1,2]); axes[r,5].set_xticklabels(["pre","early","late"])
        axes[r,5].set_ylim(0,1)
        axes[r,5].set_title("lesion mean")
    plt.tight_layout()
    fig_path = FIG_DIR / "01_dce_channels_mask_overlay_check.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_path)

    sample_rows = train_df.sample(n=min(4, len(train_df)), random_state=123)
    fig, axes = plt.subplots(len(sample_rows), 4, figsize=(16, 4*len(sample_rows)))
    if len(sample_rows) == 1: axes = np.expand_dims(axes, 0)
    for r, (_, row) in enumerate(sample_rows.iterrows()):
        path = resolve_npz_path(row["npz_path"])
        z = np.load(path)
        img = z["image"].astype(np.float32)
        msk = (z["mask"] > 0).astype(np.uint8)
        axes[r,0].imshow(overlay_rgb_on_gray(img[...,1], msk)); axes[r,0].set_title("Original early+mask"); axes[r,0].axis("off")
        for c in range(1,4):
            aug = train_aug(image=(img*255).astype(np.uint8), mask=msk)
            aimg = aug["image"].astype(np.float32)/255.0
            amsk = (aug["mask"]>0).astype(np.uint8)
            axes[r,c].imshow(overlay_rgb_on_gray(aimg[...,1], amsk)); axes[r,c].set_title(f"Light aug {c}"); axes[r,c].axis("off")
    plt.tight_layout()
    fig_path = FIG_DIR / "02_light_augmentation_check.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_path)


## Step 6 — Models


In [ ]:


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        self.e1=ConvBlock(in_ch,base)
        self.e2=ConvBlock(base,base*2)
        self.e3=ConvBlock(base*2,base*4)
        self.e4=ConvBlock(base*4,base*8)
        self.pool=nn.MaxPool2d(2)
        self.center=ConvBlock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,2); self.d4=ConvBlock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,2); self.d3=ConvBlock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,2); self.d2=ConvBlock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,2); self.d1=ConvBlock(base*2,base)
        self.out=nn.Conv2d(base,out_ch,1)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); e4=self.e4(self.pool(e3))
        c=self.center(self.pool(e4))
        d4=self.d4(torch.cat([self.u4(c),e4],1))
        d3=self.d3(torch.cat([self.u3(d4),e3],1))
        d2=self.d2(torch.cat([self.u2(d3),e2],1))
        d1=self.d1(torch.cat([self.u1(d2),e1],1))
        return self.out(d1)

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g=nn.Sequential(nn.Conv2d(F_g,F_int,1,bias=True), nn.BatchNorm2d(F_int))
        self.W_x=nn.Sequential(nn.Conv2d(F_l,F_int,1,bias=True), nn.BatchNorm2d(F_int))
        self.psi=nn.Sequential(nn.Conv2d(F_int,1,1,bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x):
        psi=self.relu(self.W_g(g)+self.W_x(x))
        psi=self.psi(psi)
        return x*psi

class AttentionUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        self.e1=ConvBlock(in_ch,base)
        self.e2=ConvBlock(base,base*2)
        self.e3=ConvBlock(base*2,base*4)
        self.e4=ConvBlock(base*4,base*8)
        self.pool=nn.MaxPool2d(2)
        self.center=ConvBlock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,2); self.a4=AttentionGate(base*8,base*8,base*4); self.d4=ConvBlock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,2); self.a3=AttentionGate(base*4,base*4,base*2); self.d3=ConvBlock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,2); self.a2=AttentionGate(base*2,base*2,base); self.d2=ConvBlock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,2); self.a1=AttentionGate(base,base,base//2); self.d1=ConvBlock(base*2,base)
        self.out=nn.Conv2d(base,out_ch,1)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); e4=self.e4(self.pool(e3))
        c=self.center(self.pool(e4))
        u4=self.u4(c); d4=self.d4(torch.cat([u4,self.a4(u4,e4)],1))
        u3=self.u3(d4); d3=self.d3(torch.cat([u3,self.a3(u3,e3)],1))
        u2=self.u2(d3); d2=self.d2(torch.cat([u2,self.a2(u2,e2)],1))
        u1=self.u1(d2); d1=self.d1(torch.cat([u1,self.a1(u1,e1)],1))
        return self.out(d1)

class SwinTinyUNet(nn.Module):
    def __init__(self, out_ch=1):
        super().__init__()
        if not TORCHVISION_OK:
            raise RuntimeError("torchvision.models.swin_t indisponible")
        self.swin = swin_t(weights=None)
        self.features = self.swin.features
        self.center = ConvBlock(768,512)
        self.up3 = nn.ConvTranspose2d(512,384,2,2); self.dec3=ConvBlock(384+384,256)
        self.up2 = nn.ConvTranspose2d(256,192,2,2); self.dec2=ConvBlock(192+192,128)
        self.up1 = nn.ConvTranspose2d(128,96,2,2); self.dec1=ConvBlock(96+96,64)
        self.up0 = nn.ConvTranspose2d(64,32,2,2); self.dec0=ConvBlock(32,32)
        self.up_final = nn.ConvTranspose2d(32,32,2,2)
        self.out = nn.Conv2d(32,out_ch,1)
    def _to_nchw(self, x):
        if x.ndim == 4 and x.shape[1] not in [96,192,384,768]:
            return x.permute(0,3,1,2).contiguous()
        return x
    def forward(self, x):
        feats=[]
        y=x
        for i, layer in enumerate(self.features):
            y=layer(y)
            if i in [1,3,5,7]:
                feats.append(self._to_nchw(y))
        if len(feats) != 4:
            raise RuntimeError(f"Expected 4 Swin features, got {len(feats)}")
        f1,f2,f3,f4 = feats
        c=self.center(f4)
        d3=self.dec3(torch.cat([self.up3(c),f3],1))
        d2=self.dec2(torch.cat([self.up2(d3),f2],1))
        d1=self.dec1(torch.cat([self.up1(d2),f1],1))
        d0=self.dec0(self.up0(d1))
        out=self.out(self.up_final(d0))
        if out.shape[-2:] != x.shape[-2:]:
            out=F.interpolate(out, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return out


def build_model(name):
    if name == "unet": return UNet(in_ch=3, out_ch=1)
    if name == "attention_unet": return AttentionUNet(in_ch=3, out_ch=1)
    if name == "swin_tiny_unet": return SwinTinyUNet(out_ch=1)
    raise ValueError(name)


def count_params(model):
    total=sum(p.numel() for p in model.parameters())
    train=sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train

param_rows=[]
for name in MODELS_TO_RUN:
    m=build_model(name).to(device)
    with torch.no_grad():
        y=m(torch.randn(1,3,IMAGE_SIZE,IMAGE_SIZE,device=device))
    total, train=count_params(m)
    print(name, "output", tuple(y.shape), "params", total)
    param_rows.append({"model":name,"params_total":total,"params_trainable":train})
    del m
    torch.cuda.empty_cache()

pd.DataFrame(param_rows).to_csv(LOG_DIR / "model_parameter_counts.csv", index=False)


## Step 7 — Loss, metrics, and surface distances


In [ ]:


class DiceBCETverskyLoss(nn.Module):
    def __init__(self, bce_weight=1.0, dice_weight=1.0, tversky_weight=1.0, alpha=0.7, beta=0.3, gamma=0.75, eps=1e-6):
        super().__init__()
        self.bce_weight=bce_weight; self.dice_weight=dice_weight; self.tversky_weight=tversky_weight
        self.alpha=alpha; self.beta=beta; self.gamma=gamma; self.eps=eps
        self.bce=nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        probs=torch.sigmoid(logits)
        bce=self.bce(logits, targets)
        dims=(1,2,3)
        inter=(probs*targets).sum(dims)
        denom=probs.sum(dims)+targets.sum(dims)
        dice=1 - ((2*inter+self.eps)/(denom+self.eps)).mean()
        tp=inter
        fp=(probs*(1-targets)).sum(dims)
        fn=((1-probs)*targets).sum(dims)
        tversky=(tp+self.eps)/(tp+self.alpha*fn+self.beta*fp+self.eps)
        ft=((1-tversky)**self.gamma).mean()
        return self.bce_weight*bce + self.dice_weight*dice + self.tversky_weight*ft

criterion = DiceBCETverskyLoss()


def binary_metrics_np(pred, true, eps=1e-7):
    pred=pred.astype(bool); true=true.astype(bool)
    tp=np.logical_and(pred,true).sum()
    fp=np.logical_and(pred,~true).sum()
    fn=np.logical_and(~pred,true).sum()
    tn=np.logical_and(~pred,~true).sum()
    dice=(2*tp+eps)/(2*tp+fp+fn+eps)
    iou=(tp+eps)/(tp+fp+fn+eps)
    prec=(tp+eps)/(tp+fp+eps)
    rec=(tp+eps)/(tp+fn+eps)
    return dice, iou, prec, rec, tp, fp, fn, tn


def parse_spacing_mm(s):
    try:
        vals=[float(x) for x in str(s).replace(",",";").split(";") if x.strip()]
        if len(vals)>=2:
            return vals[0], vals[1]
    except Exception:
        pass
    return 1.0, 1.0


def effective_spacing_2d(spacing_str, crop_size_native):
    sy, sx = parse_spacing_mm(spacing_str)
    try:
        scale=float(crop_size_native)/float(IMAGE_SIZE)
    except Exception:
        scale=1.0
    return (sy*scale, sx*scale)


def surface_distances(pred, true, spacing=(1.0,1.0)):
    pred=pred.astype(bool); true=true.astype(bool)
    diag=math.sqrt((pred.shape[0]*spacing[0])**2 + (pred.shape[1]*spacing[1])**2)
    if pred.sum()==0 or true.sum()==0:
        return np.array([diag], dtype=np.float32)
    pred_s = np.logical_xor(pred, binary_erosion(pred))
    true_s = np.logical_xor(true, binary_erosion(true))
    if pred_s.sum()==0: pred_s=pred
    if true_s.sum()==0: true_s=true
    dt_true = distance_transform_edt(~true_s, sampling=spacing)
    dt_pred = distance_transform_edt(~pred_s, sampling=spacing)
    d1 = dt_true[pred_s]
    d2 = dt_pred[true_s]
    if len(d1)==0 or len(d2)==0:
        return np.array([diag], dtype=np.float32)
    return np.concatenate([d1,d2]).astype(np.float32)


def hd95_asd(pred, true, spacing=(1.0,1.0)):
    d=surface_distances(pred,true,spacing)
    return float(np.percentile(d,95)), float(np.mean(d))


## Step 8 — Training and evaluation


In [ ]:


def make_optimizer(model, model_name):
    if model_name == "swin_tiny_unet":
        enc=[]; dec=[]
        for n,p in model.named_parameters():
            if not p.requires_grad: continue
            if n.startswith("swin") or n.startswith("features"):
                enc.append(p)
            else:
                dec.append(p)
        return torch.optim.AdamW([
            {"params": enc, "lr": LR_ENCODER},
            {"params": dec, "lr": LR_DECODER},
        ], weight_decay=WEIGHT_DECAY)
    else:
        return torch.optim.AdamW(model.parameters(), lr=LR_DECODER, weight_decay=WEIGHT_DECAY)


def make_scheduler(optimizer, max_epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, max_epochs-WARMUP_EPOCHS), eta_min=1e-6)


def set_warmup_lr(optimizer, epoch, base_lrs):
    factor = min(1.0, float(epoch+1)/float(max(1,WARMUP_EPOCHS)))
    for group, base_lr in zip(optimizer.param_groups, base_lrs):
        group["lr"] = base_lr * factor


def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    losses=[]
    for batch in tqdm(loader, desc="train", leave=False):
        x=batch["image"].to(device, non_blocking=True)
        y=batch["mask"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device.type=="cuda")):
            logits=model(x)
            loss=criterion(logits,y)
        scaler.scale(loss).backward()
        if GRAD_CLIP is not None:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else np.nan

@torch.no_grad()
def validate_loss_and_dice(model, loader, threshold=0.5):
    model.eval()
    losses=[]; dices=[]
    for batch in tqdm(loader, desc="val", leave=False):
        x=batch["image"].to(device, non_blocking=True)
        y=batch["mask"].to(device, non_blocking=True)
        logits=model(x)
        loss=criterion(logits,y)
        probs=torch.sigmoid(logits)
        pred=(probs>=threshold).float()
        dims=(1,2,3)
        inter=(pred*y).sum(dims)
        denom=pred.sum(dims)+y.sum(dims)
        dice=((2*inter+1e-7)/(denom+1e-7)).detach().cpu().numpy()
        losses.append(float(loss.detach().cpu()))
        dices.extend(dice.tolist())
    return float(np.mean(losses)), float(np.mean(dices))

@torch.no_grad()
def select_threshold_on_validation(model, loader, thresholds=THRESHOLDS):
    model.eval()
    sums={t:0.0 for t in thresholds}; counts={t:0 for t in thresholds}
    for batch in tqdm(loader, desc="threshold sweep val", leave=False):
        x=batch["image"].to(device, non_blocking=True)
        y=batch["mask"].cpu().numpy()[:,0].astype(bool)
        probs=torch.sigmoid(model(x)).detach().cpu().numpy()[:,0]
        for t in thresholds:
            pred=probs>=t
            for i in range(pred.shape[0]):
                d, *_ = binary_metrics_np(pred[i], y[i])
                sums[t]+=d; counts[t]+=1
    means={t: sums[t]/max(1,counts[t]) for t in thresholds}
    best_t=max(means, key=means.get)
    return best_t, means

@torch.no_grad()
def evaluate_model(model, loader, threshold, split_name, compute_distances=True):
    model.eval()
    rows=[]
    for batch in tqdm(loader, desc=f"eval {split_name}", leave=False):
        x=batch["image"].to(device, non_blocking=True)
        y=batch["mask"].cpu().numpy()[:,0].astype(bool)
        probs=torch.sigmoid(model(x)).detach().cpu().numpy()[:,0]
        metas=batch["meta"]
        pred=probs>=threshold
        bs=pred.shape[0]
        for i in range(bs):
            dice,iou,prec,rec,tp,fp,fn,tn = binary_metrics_np(pred[i], y[i])
            empty_pred = int(pred[i].sum()==0)
            if compute_distances:
                spacing=effective_spacing_2d(metas["spacing_mm"][i], metas["crop_size_native"][i])
                hd,asd=hd95_asd(pred[i], y[i], spacing=spacing)
            else:
                hd,asd=np.nan,np.nan
            rows.append({
                "split": split_name,
                "sample_id": metas["sample_id"][i],
                "dataset": metas["dataset"][i],
                "patient_id": metas["patient_id"][i],
                "dice": dice, "iou": iou, "precision": prec, "recall": rec,
                "hd95": hd, "asd": asd, "empty_pred": empty_pred,
                "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
            })
    return pd.DataFrame(rows)


def aggregate_metrics(per_sample_df):
    out={}
    for col in ["dice","iou","precision","recall","hd95","asd","empty_pred"]:
        out[col] = float(per_sample_df[col].mean()) if col in per_sample_df.columns else np.nan
    out["n"] = int(len(per_sample_df))
    return out


## Step 9 — Main loop


In [ ]:

all_seed_summaries = []
training_log = []

def find_resume_checkpoint(run_name):
    """Prefer local last checkpoint, then checkpoints supplied as Kaggle input."""
    local_last = CHECKPOINT_DIR / f"{run_name}_last.pt"
    if local_last.exists():
        return local_last
    for d in RESUME_INPUT_DIRS:
        d = Path(d)
        candidates = [
            d / "checkpoints" / f"{run_name}_last.pt",
            d / f"{run_name}_last.pt",
            d / "ROI_MRI_TrackB_Phase2_RESULTS" / "checkpoints" / f"{run_name}_last.pt",
        ]
        for p in candidates:
            if p.exists():
                return p
    if Path("/kaggle/input").exists():
        hits = list(Path("/kaggle/input").rglob(f"{run_name}_last.pt"))
        if hits:
            return hits[0]
    return None

def save_resume_zip():
    """Create a small resume zip containing logs + best/last checkpoints for the current run."""
    if not CREATE_RESUME_ZIP:
        return None
    resume_zip = OUT_DIR.parent / f"ROI_MRI_TrackB_Phase2_RESUME_{EXPERIMENT_MODEL}_seed{EXPERIMENT_SEED}.zip"
    run_name = f"{EXPERIMENT_MODEL}_seed{EXPERIMENT_SEED}"
    include = [
        OUT_DIR / "training_log.csv",
        OUT_DIR / "run_status.json",
        CHECKPOINT_DIR / f"{run_name}_last.pt",
        CHECKPOINT_DIR / f"{run_name}_best.pt",
        METRIC_DIR / f"{run_name}_summary.json",
        METRIC_DIR / "all_trackb_seed_summaries.csv",
    ]
    with zipfile.ZipFile(resume_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in include:
            if p.exists() and p.is_file():
                z.write(p, arcname=str(p.relative_to(OUT_DIR.parent)))
        for d in [LOG_DIR, FIG_DIR]:
            if d.exists():
                for p in d.rglob("*"):
                    if p.is_file():
                        z.write(p, arcname=str(p.relative_to(OUT_DIR.parent)))
    print("Saved resume ZIP:", resume_zip)
    return resume_zip

if RUN_TRAINING or RUN_EVALUATION_ONLY:
    for model_name in MODELS_TO_RUN:
        for seed in SEEDS:
            seed_everything(seed)
            run_name = f"{model_name}_seed{seed}"
            print("\n" + "="*80)
            print("RUN:", run_name)
            print("Single-run resumable mode")
            print("="*80)

            ckpt_best = CHECKPOINT_DIR / f"{run_name}_best.pt"
            ckpt_last = CHECKPOINT_DIR / f"{run_name}_last.pt"
            ckpt_final_eval = CHECKPOINT_DIR / f"{run_name}_final_eval.pt"
            metrics_path = METRIC_DIR / f"{run_name}_per_sample_metrics.csv"
            summary_path = METRIC_DIR / f"{run_name}_summary.json"

            model = build_model(model_name).to(device)
            optimizer = make_optimizer(model, model_name)
            scheduler = make_scheduler(optimizer, MAX_EPOCHS)
            base_lrs = [g["lr"] for g in optimizer.param_groups]
            scaler = GradScaler(enabled=(device.type=="cuda"))

            best_val_loss = np.inf
            best_epoch = -1
            wait = 0
            start_epoch = 0

            resume_ckpt = find_resume_checkpoint(run_name)
            if resume_ckpt is not None and not RUN_EVALUATION_ONLY:
                print("Resuming from:", resume_ckpt)
                state = torch.load(resume_ckpt, map_location=device)
                model.load_state_dict(state["model_state"], strict=True)
                if "optimizer_state" in state:
                    optimizer.load_state_dict(state["optimizer_state"])
                if "scheduler_state" in state:
                    try:
                        scheduler.load_state_dict(state["scheduler_state"])
                    except Exception as e:
                        print("WARNING: scheduler state not restored:", e)
                if "scaler_state" in state and device.type == "cuda":
                    try:
                        scaler.load_state_dict(state["scaler_state"])
                    except Exception as e:
                        print("WARNING: scaler state not restored:", e)
                best_val_loss = float(state.get("best_val_loss", np.inf))
                best_epoch = int(state.get("best_epoch", -1))
                wait = int(state.get("wait", 0))
                start_epoch = int(state.get("epoch", -1)) + 1
                print(f"Resume state: start_epoch={start_epoch}, best_epoch={best_epoch}, best_val_loss={best_val_loss}, wait={wait}")

            log_path = OUT_DIR / "training_log.csv"
            if log_path.exists():
                try:
                    prev_log = pd.read_csv(log_path)
                    training_log = prev_log.to_dict(orient="records")
                    print("Recovered previous training_log rows:", len(training_log))
                except Exception as e:
                    print("WARNING: could not recover training_log.csv:", e)

            clean_stop = False

            if not RUN_EVALUATION_ONLY:
                for epoch in range(start_epoch, MAX_EPOCHS):
                    epoch_start = time.time()
                    elapsed_min = (time.time() - RUN_STARTED_AT) / 60.0
                    if AUTO_STOP_BEFORE_TIMEOUT and elapsed_min >= STOP_AFTER_MINUTES:
                        print(f"Time guard reached before epoch {epoch}: {elapsed_min:.1f} min. Saving and stopping cleanly.")
                        clean_stop = True
                        break

                    if epoch < WARMUP_EPOCHS:
                        set_warmup_lr(optimizer, epoch, base_lrs)

                    train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
                    val_loss, val_dice50 = validate_loss_and_dice(model, val_loader, threshold=0.5)

                    if epoch >= WARMUP_EPOCHS:
                        scheduler.step()

                    lrs = [g["lr"] for g in optimizer.param_groups]
                    improved = val_loss < best_val_loss - 1e-5
                    if improved:
                        best_val_loss = val_loss
                        best_epoch = epoch
                        wait = 0
                        torch.save({
                            "model_state": model.state_dict(),
                            "model": model_name,
                            "seed": seed,
                            "epoch": epoch,
                            "val_loss": float(val_loss),
                            "best_val_loss": float(best_val_loss),
                            "best_epoch": int(best_epoch),
                        }, ckpt_best)
                    else:
                        wait += 1

                    torch.save({
                        "model_state": model.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "scheduler_state": scheduler.state_dict(),
                        "scaler_state": scaler.state_dict() if device.type == "cuda" else None,
                        "model": model_name,
                        "seed": seed,
                        "epoch": epoch,
                        "train_loss": float(train_loss),
                        "val_loss": float(val_loss),
                        "val_dice_0p5": float(val_dice50),
                        "best_val_loss": float(best_val_loss),
                        "best_epoch": int(best_epoch),
                        "wait": int(wait),
                        "max_epochs": int(MAX_EPOCHS),
                    }, ckpt_last)

                    row = {
                        "run_name": run_name,
                        "model": model_name,
                        "seed": seed,
                        "epoch": epoch,
                        "train_loss": train_loss,
                        "val_loss": val_loss,
                        "val_dice_0p5": val_dice50,
                        "best_val_loss": best_val_loss,
                        "best_epoch": best_epoch,
                        "wait": wait,
                        "improved": bool(improved),
                        "lr_groups": ";".join([str(x) for x in lrs]),
                        "sec": time.time() - epoch_start,
                    }
                    training_log.append(row)
                    pd.DataFrame(training_log).to_csv(log_path, index=False)

                    with open(OUT_DIR / "run_status.json", "w", encoding="utf-8") as f:
                        json.dump({
                            "run_name": run_name,
                            "model": model_name,
                            "seed": seed,
                            "last_epoch": int(epoch),
                            "best_epoch": int(best_epoch),
                            "best_val_loss": float(best_val_loss),
                            "wait": int(wait),
                            "status": "training",
                            "elapsed_minutes": float((time.time()-RUN_STARTED_AT)/60.0),
                        }, f, indent=2)

                    print(
                        f"epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f} "
                        f"| dice@0.5 {val_dice50:.4f} | best {best_val_loss:.4f}@{best_epoch} | wait {wait}"
                    )

                    if wait >= PATIENCE:
                        print("Early stopping.")
                        break

                save_resume_zip()

            # Evaluate only when a best checkpoint is available.
            if not ckpt_best.exists():
                print("No best checkpoint available yet. This run stopped before evaluation.")
                with open(OUT_DIR / "run_status.json", "w", encoding="utf-8") as f:
                    json.dump({
                        "run_name": run_name,
                        "status": "stopped_without_best_checkpoint",
                        "message": "Resume this run using the generated resume ZIP or run longer.",
                    }, f, indent=2)
                continue

            state = torch.load(ckpt_best, map_location=device)
            model.load_state_dict(state["model_state"], strict=True)
            model.eval()
            torch.save({
                "model_state": model.state_dict(),
                "model": model_name,
                "seed": seed,
                "best_epoch": int(state.get("epoch", -1)),
                "best_val_loss": float(state.get("val_loss", np.nan)),
            }, ckpt_final_eval)

            best_t, val_threshold_scores = select_threshold_on_validation(model, val_loader, thresholds=THRESHOLDS)
            print("Selected threshold on validation:", best_t, "val dice:", val_threshold_scores[best_t])
            with open(METRIC_DIR / f"{run_name}_threshold_sweep.json", "w", encoding="utf-8") as f:
                json.dump({str(k): float(v) for k, v in val_threshold_scores.items()}, f, indent=2)

            split_metrics = []
            for split_name, loader in [("validation", val_loader), ("test", test_loader), ("external", ext_loader)]:
                per = evaluate_model(model, loader, threshold=best_t, split_name=split_name, compute_distances=True)
                per["model"] = model_name
                per["seed"] = seed
                per["threshold"] = best_t
                per["run_name"] = run_name
                split_metrics.append(per)
                agg = aggregate_metrics(per)
                print(split_name, agg)

            per_all = pd.concat(split_metrics, ignore_index=True)
            per_all.to_csv(metrics_path, index=False)

            for split_name in ["validation", "test", "external"]:
                sub = per_all[per_all["split"] == split_name]
                agg = aggregate_metrics(sub)
                row = {
                    "run_name": run_name,
                    "model": model_name,
                    "seed": seed,
                    "split": split_name,
                    "threshold": best_t,
                    "best_epoch": int(state.get("epoch", -1)),
                    "best_val_loss": float(state.get("val_loss", np.nan)),
                    **agg,
                }
                all_seed_summaries.append(row)

            summary_csv = METRIC_DIR / "all_trackb_seed_summaries.csv"
            if summary_csv.exists():
                old = pd.read_csv(summary_csv)
                new = pd.DataFrame(all_seed_summaries)
                merged = pd.concat([old, new], ignore_index=True)
                merged = merged.drop_duplicates(subset=["run_name", "split"], keep="last")
            else:
                merged = pd.DataFrame(all_seed_summaries)
            merged.to_csv(summary_csv, index=False)

            with open(summary_path, "w", encoding="utf-8") as f:
                json.dump([r for r in all_seed_summaries if r["run_name"] == run_name], f, indent=2, ensure_ascii=False)

            with open(OUT_DIR / "run_status.json", "w", encoding="utf-8") as f:
                json.dump({
                    "run_name": run_name,
                    "status": "completed_and_evaluated",
                    "best_epoch": int(state.get("epoch", -1)),
                    "selected_threshold": float(best_t),
                }, f, indent=2)

            save_resume_zip()
            del model
            torch.cuda.empty_cache()
else:
    print("RUN_TRAINING=False and RUN_EVALUATION_ONLY=False: training loop not executed.")


## Step 10 — Final aggregation and report


In [ ]:


summary_csv = METRIC_DIR / "all_trackb_seed_summaries.csv"
if summary_csv.exists():
    s = pd.read_csv(summary_csv)
    display(s.head())

    agg_rows=[]
    for (model, split), g in s.groupby(["model","split"]):
        row={"model":model,"split":split,"n_seeds":g["seed"].nunique()}
        for col in ["threshold","dice","iou","precision","recall","hd95","asd","empty_pred"]:
            row[f"{col}_mean"] = float(g[col].mean())
            row[f"{col}_std"] = float(g[col].std(ddof=1)) if len(g)>1 else 0.0
        agg_rows.append(row)
    agg=pd.DataFrame(agg_rows).sort_values(["split","dice_mean"], ascending=[True,False])
    agg.to_csv(OUT_DIR / "trackb_results.csv", index=False)
    display(agg)

    wide=[]
    for model, gm in agg.groupby("model"):
        row={"model":model}
        for _, r in gm.iterrows():
            split=r["split"]
            for col in ["dice","iou","precision","recall","hd95","asd","empty_pred","threshold"]:
                row[f"{split}_{col}_mean"] = r[f"{col}_mean"]
                row[f"{split}_{col}_std"] = r[f"{col}_std"]
        wide.append(row)
    wide=pd.DataFrame(wide)
    wide.to_csv(OUT_DIR / "trackb_results_wide.csv", index=False)
    display(wide)

    # Select the best configuration using validation Dice only.
    val_agg=agg[agg["split"]=="validation"].sort_values("dice_mean", ascending=False)
    best_model = val_agg.iloc[0]["model"] if len(val_agg) else None

    lines=[]
    lines.append("# Track B IRM ROI 256 DCE — Training Results")
    lines.append("")
    lines.append("## Protocol")
    lines.append("")
    lines.append("- Dataset: ROI_MRI_Crops_256_v1")
    lines.append("- Input: 256×256×3 DCE channels `[pre, early, late]`")
    lines.append("- Splits: train / validation / test internal, Duke external sealed")
    lines.append("- Threshold selected only on validation, then frozen for test/external")
    lines.append("- Metrics: Dice, IoU, precision, recall, HD95, ASD, empty prediction rate")
    lines.append("")
    lines.append("## Aggregated results")
    lines.append("")
    lines.append(agg.to_markdown(index=False))
    lines.append("")
    lines.append("## Wide table")
    lines.append("")
    lines.append(wide.to_markdown(index=False))
    lines.append("")
    lines.append("## Validation-selected recommendation")
    lines.append("")
    if best_model is not None:
        lines.append(f"- Best model by validation Dice: **{best_model}**")
        lines.append("- This choice is based only on validation performance; Duke external remains sealed for selection.")
    else:
        lines.append("- No completed training results found.")

    md_path=OUT_DIR / "trackb_results.md"
    with open(md_path,"w",encoding="utf-8") as f:
        f.write("\n".join(lines))
    print("Saved:", md_path)
else:
    print("No training summary found yet:", summary_csv)


## Step 11 — Light ZIP


In [ ]:

if CREATE_LIGHT_ZIP:
    zip_path = OUT_DIR.parent / f"ROI_MRI_TrackB_Phase2_RESULTS_LIGHT_{EXPERIMENT_MODEL}_seed{EXPERIMENT_SEED}.zip"
    include_files = [
        OUT_DIR / "trackb_results.csv",
        OUT_DIR / "trackb_results_wide.csv",
        OUT_DIR / "trackb_results.md",
        OUT_DIR / "training_log.csv",
        OUT_DIR / "run_status.json",
    ]
    include_dirs = [LOG_DIR, FIG_DIR]
    extra = [METRIC_DIR / "all_trackb_seed_summaries.csv"]

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in include_files + extra:
            if p.exists() and p.is_file():
                z.write(p, arcname=str(p.relative_to(OUT_DIR.parent)))
        for d in include_dirs:
            if d.exists():
                for p in d.rglob("*"):
                    if p.is_file():
                        z.write(p, arcname=str(p.relative_to(OUT_DIR.parent)))
    print("Saved light ZIP:", zip_path)

if CREATE_RESUME_ZIP:
    run_name = f"{EXPERIMENT_MODEL}_seed{EXPERIMENT_SEED}"
    resume_zip = OUT_DIR.parent / f"ROI_MRI_TrackB_Phase2_RESUME_{EXPERIMENT_MODEL}_seed{EXPERIMENT_SEED}.zip"
    include = [
        OUT_DIR / "training_log.csv",
        OUT_DIR / "run_status.json",
        CHECKPOINT_DIR / f"{run_name}_last.pt",
        CHECKPOINT_DIR / f"{run_name}_best.pt",
        CHECKPOINT_DIR / f"{run_name}_final_eval.pt",
        METRIC_DIR / f"{run_name}_summary.json",
        METRIC_DIR / f"{run_name}_threshold_sweep.json",
        METRIC_DIR / "all_trackb_seed_summaries.csv",
    ]
    with zipfile.ZipFile(resume_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in include:
            if p.exists() and p.is_file():
                z.write(p, arcname=str(p.relative_to(OUT_DIR.parent)))
        for d in [LOG_DIR, FIG_DIR]:
            if d.exists():
                for p in d.rglob("*"):
                    if p.is_file():
                        z.write(p, arcname=str(p.relative_to(OUT_DIR.parent)))
    print("Saved resume ZIP:", resume_zip)
